# Usecase 4: EMP environment classification data preparation

Prepares the Earth Microbiome Project (EMP) release 1 data for the EMPO-3 environment classification usecase following [Thompson et al. 2017](https://doi.org/10.1038/nature24621). Target: `empo_3`. EMP publishes a fixed split — the 2,000-sample subset (`subset_2k`) trains, all other QC-filtered samples (21,828) are held out — so ritme's `split_train_test` is bypassed and the splits are written directly to:
- `data_splits_u4/`: raw Deblur 90-bp counts as relative abundances
- `data_splits_u4_rare5000/`: the same table rarefied to 5,000 reads per sample, also as relative abundances

`n4_original_setup.ipynb` reads **both**; keep both.

Two classes of the published 17 (`Hypersaline (saline)`, `Surface (saline)`) have no held-out samples at all, so they cannot be evaluated and make a macro one-vs-rest AUROC over the full class list undefined. Their 130 training samples are dropped, leaving **15 classes and 1,870 training samples** — a documented deviation from the paper's 2,000-sample subset. EMP's own published EMPO-3 confusion matrix (`data/random-forest/empo_3_cm.txt`) likewise covers 15 classes.

Apart from the classes above, the full feature space is kept: only features that are zero in every retained sample are removed, which is ritme's own rule.

This notebook can be run in the following conda environment (the last command must be launched from the root of this repository):
```shell
mamba env create -f environment_prep_data.yml
conda activate ritme_examples_prep_data
pip install papermill
pip install -e .
```

The download needs internet access. Where compute nodes are offline, fetch once from a login node (on Euler: `module load eth_proxy`) from the repository root before executing the notebook:
```shell
src/fetch_emp.sh data/u4_emp
```
Execute the notebook headless from this directory. It writes four pickles totalling ~118 GiB and holds one dense 21,828-row table in memory at a time:
```shell
python -m ipykernel install --user --name ritme_examples_prep_data
sbatch --cpus-per-task=16 --mem-per-cpu=16G --time=12:00:00 --wrap \
  "papermill -k ritme_examples_prep_data n1_data.ipynb n1_data.ipynb"
```

## Setup

In [1]:
import json
import os
import subprocess

import numpy as np
import pandas as pd

from src.prepare_u4 import (
    build_metadata,
    build_split_tables,
    compare_rarefied_tables,
    extract_taxonomy,
    process_peak_rss_gib,
)

%load_ext autoreload
%autoreload 2

In [2]:
######## USER INPUTS ########
# where all EMP downloads and the derived metadata/taxonomy are stored
path_to_data = "../../data/u4_emp"
# ritme-ready splits: raw Deblur table and the rarefied table
path_splits_raw = "data_splits_u4"
path_splits_rare = "data_splits_u4_rare5000"

# published membership of the EMP split, enforced before any sample is dropped
published_counts = {"train": 2000, "test": 21828}
# classes without held-out samples, dropped; and what remains afterwards
expected_absent_from_test = ["Hypersaline (saline)", "Surface (saline)"]
expected_n_dropped = 130
expected_n = {"train": 1870, "test": 21828}
expected_n_classes = 15
# observation counts the two BIOM tables declare
declared_n_features = {"raw": 317314, "rare5000": 307572}
n_samples_total = sum(expected_n.values())
# rows per chunk in the sanity checks; 200 x ~315k float64 is ~0.5 GB
check_chunk = 200
######## END USER INPUTS #####

## Fetch EMP release 1 data

Deblur 90-bp tables (QC-filtered and 2,000-sample subset, each raw and rarefied to 5,000 reads) plus the mapping file. Files already present with the expected size are skipped.

In [3]:
subprocess.run(["../../src/fetch_emp.sh", path_to_data], check=True)
for f in sorted(os.listdir(path_to_data)):
    print(f"{os.path.getsize(os.path.join(path_to_data, f)) / 1e6:8.1f} MB  {f}")

[skip] ../../data/u4_emp/emp_deblur_90bp.subset_2k.biom already present
[skip] ../../data/u4_emp/emp_deblur_90bp.subset_2k.rare_5000.biom already present
[skip] ../../data/u4_emp/emp_deblur_90bp.qc_filtered.biom already present
[skip] ../../data/u4_emp/emp_deblur_90bp.qc_filtered.rare_5000.biom already present
[skip] ../../data/u4_emp/emp_qiime_mapping_release1.tsv already present
   266.7 MB  emp_deblur_90bp.qc_filtered.biom
   176.0 MB  emp_deblur_90bp.qc_filtered.rare_5000.biom
   115.5 MB  emp_deblur_90bp.subset_2k.biom
    56.4 MB  emp_deblur_90bp.subset_2k.rare_5000.biom
    21.9 MB  emp_qiime_mapping_release1.tsv
    28.7 MB  feature_ids.txt
    28.0 MB  feature_ids_rare5000.txt
     1.7 MB  md_emp.tsv
     0.0 MB  table_stats.json
    47.3 MB  taxonomy_emp.tsv


## Split membership and metadata

`train` = `subset_2k`, `test` = `qc_filtered` minus `subset_2k`, then the classes without held-out samples are dropped.

In [4]:
md_emp, split_report = build_metadata(
    os.path.join(path_to_data, "emp_qiime_mapping_release1.tsv"),
    os.path.join(path_to_data, "md_emp.tsv"),
    published_counts=published_counts,
)
print(json.dumps(split_report, indent=2))
pd.crosstab(md_emp["empo_3"], md_emp["split"])

{
  "published_counts": {
    "test": 21828,
    "train": 2000
  },
  "n_classes_published": 17,
  "classes_absent_from_test": [
    "Hypersaline (saline)",
    "Surface (saline)"
  ],
  "n_samples_dropped": 130,
  "counts": {
    "test": 21828,
    "train": 1870
  },
  "n_classes": 15
}


split,test,train
empo_3,,
Aerosol (non-saline),4,81
Animal corpus,200,128
Animal distal gut,4030,128
Animal proximal gut,239,128
Animal secretion,1144,113
Animal surface,2818,143
Plant corpus,2,123
Plant rhizosphere,426,128
Plant surface,1483,128


In [5]:
assert split_report["classes_absent_from_test"] == expected_absent_from_test
assert split_report["n_samples_dropped"] == expected_n_dropped
assert split_report["counts"] == expected_n, split_report["counts"]
assert split_report["n_classes"] == expected_n_classes

## Materialise the fixed split

Reproduces the preprocessing of ritme's `split_train_test` on the sparse BIOM table: the table is restricted to the split's samples, features prefixed with `F`, features that are zero across all of them removed, rows converted to relative abundances. Metadata columns precede the features as in ritme's pickles.

In [6]:
stats_raw = build_split_tables(
    os.path.join(path_to_data, "emp_deblur_90bp.qc_filtered.biom"),
    md_emp,
    path_splits_raw,
    os.path.join(path_to_data, "feature_ids.txt"),
    expected_n=expected_n,
    declared_n_features=declared_n_features["raw"],
)
stats_raw

{'biom': 'emp_deblur_90bp.qc_filtered.biom',
 'n_features_declared': 317314,
 'n_features': 311617,
 'n_features_all_zero_removed': 5697,
 'n_samples_in_split': 23698,
 'n_train': 1870,
 'read_depth_train': {'q00': 5275.0,
  'q05': 11318.300000000001,
  'q25': 25985.75,
  'q50': 48642.5,
  'q75': 77056.5,
  'q95': 157186.3,
  'q100': 589438.0,
  'mean': 61927.14812834225},
 'empo_3_counts_train': {'Aerosol (non-saline)': 81,
  'Animal corpus': 128,
  'Animal distal gut': 128,
  'Animal proximal gut': 128,
  'Animal secretion': 113,
  'Animal surface': 143,
  'Plant corpus': 123,
  'Plant rhizosphere': 128,
  'Plant surface': 128,
  'Sediment (non-saline)': 128,
  'Sediment (saline)': 128,
  'Soil (non-saline)': 129,
  'Surface (non-saline)': 128,
  'Water (non-saline)': 129,
  'Water (saline)': 128},
 'train_val.pkl_gib': 4.37,
 'n_test': 21828,
 'read_depth_test': {'q00': 5001.0,
  'q05': 9762.0,
  'q25': 22426.75,
  'q50': 40970.0,
  'q75': 60246.0,
  'q95': 116744.14999999989,
  'q1

In [7]:
stats_rare = build_split_tables(
    os.path.join(path_to_data, "emp_deblur_90bp.qc_filtered.rare_5000.biom"),
    md_emp,
    path_splits_rare,
    os.path.join(path_to_data, "feature_ids_rare5000.txt"),
    expected_n=expected_n,
    declared_n_features=declared_n_features["rare5000"],
)
stats_rare

{'biom': 'emp_deblur_90bp.qc_filtered.rare_5000.biom',
 'n_features_declared': 307572,
 'n_features': 303200,
 'n_features_all_zero_removed': 4372,
 'n_samples_in_split': 23698,
 'n_train': 1870,
 'read_depth_train': {'q00': 5000.0,
  'q05': 5000.0,
  'q25': 5000.0,
  'q50': 5000.0,
  'q75': 5000.0,
  'q95': 5000.0,
  'q100': 5000.0,
  'mean': 5000.0},
 'empo_3_counts_train': {'Aerosol (non-saline)': 81,
  'Animal corpus': 128,
  'Animal distal gut': 128,
  'Animal proximal gut': 128,
  'Animal secretion': 113,
  'Animal surface': 143,
  'Plant corpus': 123,
  'Plant rhizosphere': 128,
  'Plant surface': 128,
  'Sediment (non-saline)': 128,
  'Sediment (saline)': 128,
  'Soil (non-saline)': 129,
  'Surface (non-saline)': 128,
  'Water (non-saline)': 129,
  'Water (saline)': 128},
 'train_val.pkl_gib': 4.25,
 'n_test': 21828,
 'read_depth_test': {'q00': 5000.0,
  'q05': 5000.0,
  'q25': 5000.0,
  'q50': 5000.0,
  'q75': 5000.0,
  'q95': 5000.0,
  'q100': 5000.0,
  'mean': 5000.0},
 'emp

## Taxonomy

Greengenes lineages shipped inside the Deblur BIOM (7 ranks). Unresolved ranks are kept as bare prefixes (`o__`) and trailing unresolved ranks dropped; ritme reads both as `unknown` and sums every such feature into one column at that rank, so `n_pooled_into_unknown_at` records how much of the space collapses per aggregation level. Feature ids stay unprefixed — ritme adds the `F`.

In [8]:
tax_stats = extract_taxonomy(
    os.path.join(path_to_data, "emp_deblur_90bp.qc_filtered.biom"),
    os.path.join(path_to_data, "feature_ids.txt"),
    os.path.join(path_to_data, "taxonomy_emp.tsv"),
)
print(json.dumps(tax_stats, indent=2))
pd.read_csv(
    os.path.join(path_to_data, "taxonomy_emp.tsv"), sep="\t", index_col=0
).head()

{
  "n_features": 311617,
  "frac_kingdom_only": 0.1399,
  "n_levels_counts": {
    "1": 43590,
    "2": 36000,
    "3": 56160,
    "4": 71427,
    "5": 64446,
    "6": 35103,
    "7": 4891
  },
  "n_pooled_into_unknown_at": {
    "phylum": 43590,
    "class": 79590,
    "order": 135750,
    "family": 207177,
    "genus": 271623,
    "species": 306726
  }
}


,Taxon
Feature ID,
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTGTTTAAGCTAGCTGTGAAAGCCCCGGGCTTAAC,k__Bacteria; p__Proteobacteria; c__Gammaproteo...
TACAGAGGTCCCAAGCGTTGTTCGGATTCATTGGGCGTAAAGGGCTCGTAGGTGGCCAACTAAGTCAGACGTGAAATCCCTCGGCTTAAC,k__Bacteria
TACGAAGGGTGCAAGCGTTGTTCGGAATAACTGGGCGTAAAGCGCACGTAGGCGGGTCCGTGTGTCGGTTGTGAAATCCCTGGGCTCAAC,k__Bacteria; p__Proteobacteria; c__Deltaproteo...
TACGGAGTGTGCAAGCGTTACTCGGAATCACTGGGCATAAAGAGCACGTAGGCGGGTCACCAAGTCAGCCGTGAAAGCCCCCGGCCCAAC,k__Bacteria; p__Planctomycetes; c__028H05-P-BN-P5
TACGAGAGGTCCAAACGTTATTCGGAATTACTGGGCTTAAAGAGTTCGTAGGCGGCTAAGTAAGTGGGATGTGAAAGCCCTCGGCTCAAC,k__Bacteria; p__Planctomycetes; c__Planctomyce...


## Table statistics

Feature counts, read-depth quantiles and per-class counts for the methods text.

The last check compares EMP's separately published `subset_2k.rare_5000` table with the QC-filtered rarefied table used here. It is **not** the same rarefaction draw (see the output below), which is why both splits are derived from the single QC-filtered table rather than mixing the two.

In [9]:
rarefaction_check = compare_rarefied_tables(
    os.path.join(path_to_data, "emp_deblur_90bp.subset_2k.rare_5000.biom"),
    os.path.join(path_to_data, "emp_deblur_90bp.qc_filtered.rare_5000.biom"),
)
table_stats = {
    "split": split_report,
    "raw": stats_raw,
    "rare5000": stats_rare,
    "taxonomy": tax_stats,
    "subset_2k_rare5000_vs_qc_filtered_rare5000": rarefaction_check,
    "process_peak_rss_gib": round(process_peak_rss_gib(), 1),
}
with open(os.path.join(path_to_data, "table_stats.json"), "w") as f:
    json.dump(table_stats, f, indent=2)
rarefaction_check

{'shared_samples': 2000,
 'shared_features': 153323,
 'features_only_in_subset_table': 1679,
 'features_only_in_full_table': 154249,
 'identical_counts': False,
 'n_differing_entries': 907451}

## Sanity checks

In [10]:
assert __debug__, "run without python -O so these checks execute"

# feature counts anchored to each table's declared observation count
for arm, stats in (("raw", stats_raw), ("rare5000", stats_rare)):
    assert (
        stats["n_features"] + stats["n_features_all_zero_removed"]
        == declared_n_features[arm]
    )
    assert stats["n_samples_in_split"] == n_samples_total
    print(
        arm,
        "features:",
        stats["n_features"],
        "| all-zero removed:",
        stats["n_features_all_zero_removed"],
    )

with open(os.path.join(path_to_data, "feature_ids.txt")) as f:
    features_raw = f.read().split()
with open(os.path.join(path_to_data, "feature_ids_rare5000.txt")) as f:
    features_rare = f.read().split()
assert len(features_raw) == stats_raw["n_features"] > 0
assert len(features_rare) == stats_rare["n_features"] > 0
# rarefied features are a subset of the raw features, in the same relative order
pos = pd.Index(features_raw).get_indexer(features_rare)
assert (pos >= 0).all() and (pd.Series(pos).diff().dropna() > 0).all()

metadata_cols = ["empo_3", "empo_2", "study_id", "split"]
for splits_dir, features in (
    (path_splits_raw, features_raw),
    (path_splits_rare, features_rare),
):
    train = pd.read_pickle(os.path.join(splits_dir, "train_val.pkl"))
    test = pd.read_pickle(os.path.join(splits_dir, "test.pkl"))
    assert train.shape == (
        expected_n["train"],
        len(features) + len(metadata_cols),
    ), train.shape
    assert test.shape == (
        expected_n["test"],
        len(features) + len(metadata_cols),
    ), test.shape
    assert list(train.columns[: len(metadata_cols)]) == metadata_cols
    assert train.columns.equals(test.columns)
    assert list(train.columns[len(metadata_cols) :]) == ["F" + f for f in features]

    # membership matches the metadata, and the two splits are disjoint
    assert set(train.index) == set(md_emp.index[md_emp["split"] == "train"])
    assert set(test.index) == set(md_emp.index[md_emp["split"] == "test"])
    assert (train["split"] == "train").all() and (test["split"] == "test").all()
    assert train["empo_3"].notna().all() and test["empo_3"].notna().all()
    assert train["empo_3"].nunique() == test["empo_3"].nunique() == expected_n_classes

    # re-verify on the written artifact: rows sum to 1, and no feature is zero
    # across train and test together (individual splits legitimately have many)
    col_sum = np.zeros(len(features))
    for frame in (train, test):
        feat = frame.iloc[:, len(metadata_cols) :]
        assert (feat.dtypes == np.float64).all()
        for start in range(0, len(feat), check_chunk):
            block = feat.iloc[start : start + check_chunk].to_numpy()
            assert np.allclose(block.sum(axis=1), 1.0, atol=1e-3), (splits_dir, start)
            col_sum += block.sum(axis=0)
    assert (col_sum > 0).all(), int((col_sum == 0).sum())
    print(splits_dir, train.shape, test.shape, "verified")
    del train, test, feat

# taxonomy covers exactly the raw arm's feature columns, unprefixed
tax = pd.read_csv(os.path.join(path_to_data, "taxonomy_emp.tsv"), sep="\t", index_col=0)
assert tax.index.name == "Feature ID" and list(tax.columns) == ["Taxon"]
assert list(tax.index) == features_raw
assert tax["Taxon"].notna().all() and not tax["Taxon"].str.startswith("F").any()
# partial lineages are preserved rather than dropped
assert 0.10 < tax_stats["frac_kingdom_only"] < 0.20, tax_stats["frac_kingdom_only"]

# none of the written data is visible to git
status = subprocess.run(
    [
        "git",
        "status",
        "--porcelain",
        "--",
        path_to_data,
        path_splits_raw,
        path_splits_rare,
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout
assert not status.strip(), status
print("All sanity checks passed.")

raw features: 311617 | all-zero removed: 5697
rare5000 features: 303200 | all-zero removed: 4372


data_splits_u4 (1870, 311621) (21828, 311621) verified


data_splits_u4_rare5000 (1870, 303204) (21828, 303204) verified


All sanity checks passed.
